# Part A – Data Understanding and Business Relevance

In [1]:
from pathlib import Path
import pyarrow.parquet as pq

DATA_PATH = Path("../data/train.parquet")

parquet_file = pq.ParquetFile(DATA_PATH)

print("Số dòng:", parquet_file.metadata.num_rows)
print("Số cột:", parquet_file.metadata.num_columns)
print("Số nhóm dữ liệu:", parquet_file.num_row_groups)

print("\nSchema:")
print(parquet_file.schema_arrow)

Số dòng: 4500000
Số cột: 19
Số nhóm dữ liệu: 4500

Schema:
city_id: int64
store_id: int64
management_group_id: int64
first_category_id: int64
second_category_id: int64
third_category_id: int64
product_id: int64
dt: string
sale_amount: double
hours_sale: list<element: double>
  child 0, element: double
stock_hour6_22_cnt: int32
hours_stock_status: list<element: int64>
  child 0, element: int64
discount: double
holiday_flag: int32
activity_flag: int32
precpt: double
avg_temperature: double
avg_humidity: double
avg_wind_level: double
-- schema metadata --
huggingface: '{"info": {"features": {"city_id": {"dtype": "int64", "_type' + 1081


4.500.000 dòng.
19 cột.
4.500 nhóm dữ liệu, trung bình khoảng 1.000 dòng mỗi nhóm.
Mỗi dòng đại diện cho một sản phẩm tại một cửa hàng ở một thành phố trong một ngày (city–store–product–day).

- sale_amount là số lượng bán trong ngày sau khi chuẩn hóa, không nhất thiết là doanh thu tiền tệ.
- hours_sale chứa doanh số theo 24 giờ.
- stock_hour6_22_cnt là số giờ hết hàng từ 6:00 đến 22:00.
- hours_stock_status chứa trạng thái hết hàng theo từng giờ.
- discount = 1.0 nghĩa là không giảm giá; discount = 0.9 nghĩa là giảm 10%. Vì vậy phần trăm giảm giá phải tính là: DiscountPercent = (1 - discount) * 100

In [2]:
sample = parquet_file.read_row_group(0).to_pandas()

print(sample.head())
print(sample.dtypes)

   city_id  store_id  management_group_id  first_category_id  \
0        0         0                    0                  5   
1        0         0                    0                  5   
2        0         0                    0                  5   
3        0         0                    0                  5   
4        0         0                    0                  5   

   second_category_id  third_category_id  product_id          dt  sale_amount  \
0                   6                 65          38  2024-03-28          0.1   
1                   6                 65          38  2024-03-29          0.1   
2                   6                 65          38  2024-03-30          0.0   
3                   6                 65          38  2024-03-31          0.1   
4                   6                 65          38  2024-04-01          0.2   

                                          hours_sale  stock_hour6_22_cnt  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, ...

## A1. What Does One Row Represent?

Lệnh này chỉ đọc row group đầu tiên, không đọc toàn bộ 4,5 triệu dòng. Row group đầu tiên có khoảng 1.000 dòng.
Năm dòng đầu cho thấy:
- Thành phố: city_id = 0
- Cửa hàng: store_id = 0
- Sản phẩm: product_id = 38
- Ngày: từ 2024-03-28 đến 2024-04-01
- Doanh số quan sát được: từ 0.0 đến 0.2
- stock_hour6_22_cnt: số giờ hết hàng trong khoảng 6:00–22:00
- discount = 1.0: không giảm giá
- holiday_flag = 0: không phải ngày lễ
- hours_sale: doanh số theo từng giờ
- hours_stock_status: trạng thái hết hàng theo từng giờ
Theo tài liệu dataset, hours_sale và hours_stock_status đều có 24 giá trị, tương ứng với 24 giờ trong ngày. stock_hour6_22_cnt là số giờ hết hàng từ 6:00 đến 22:00. Dataset Card
Ý nghĩa kiểu dữ liệu
- int64, int32: các mã định danh hoặc biến cờ.
- float64: doanh số, giảm giá, nhiệt độ, độ ẩm, mưa.
- object: ngày dạng chuỗi và các cột chứa danh sách theo giờ.

In [3]:
id_columns = [
    "city_id",
    "store_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "product_id"
]

for column in id_columns:
    print(column, ":", sample[column].nunique(), "giá trị khác nhau trong mẫu")

city_id : 1 giá trị khác nhau trong mẫu
store_id : 1 giá trị khác nhau trong mẫu
management_group_id : 3 giá trị khác nhau trong mẫu
first_category_id : 6 giá trị khác nhau trong mẫu
second_category_id : 7 giá trị khác nhau trong mẫu
third_category_id : 10 giá trị khác nhau trong mẫu
product_id : 12 giá trị khác nhau trong mẫu


In [4]:
unique_values = {column: set() for column in id_columns}

for batch in parquet_file.iter_batches(
    batch_size=100_000,
    columns=id_columns
):
    batch_data = batch.to_pydict()

    for column in id_columns:
        unique_values[column].update(batch_data[column])

for column in id_columns:
    print(column, ":", len(unique_values[column]), "giá trị khác nhau")

city_id : 18 giá trị khác nhau
store_id : 898 giá trị khác nhau
management_group_id : 7 giá trị khác nhau
first_category_id : 32 giá trị khác nhau
second_category_id : 84 giá trị khác nhau
third_category_id : 233 giá trị khác nhau
product_id : 865 giá trị khác nhau


Ý nghĩa:
- Có 18 thành phố.
- Có 898 cửa hàng.
- Có 865 sản phẩm.
- Có 7 nhóm quản lý.
- Có 32 nhóm danh mục cấp 1.
- Có 84 nhóm danh mục cấp 2.
- Có 233 nhóm danh mục cấp 3.

In [5]:
print("Số giờ trong hours_sale:", len(sample.loc[0, "hours_sale"]))
print("Số giờ trong hours_stock_status:", len(sample.loc[0, "hours_stock_status"]))

print("hours_sale:", sample.loc[0, "hours_sale"])
print("hours_stock_status:", sample.loc[0, "hours_stock_status"])

Số giờ trong hours_sale: 24
Số giờ trong hours_stock_status: 24
hours_sale: [0.  0.  0.  0.  0.  0.  0.  0.1 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
 0.  0.  0.  0.  0.  0. ]
hours_stock_status: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## A2. Variable Classification

| Variable group | Variables | Business meaning |
|---|---|---|
| Identification | `city_id`, `store_id`, `management_group_id`, `first_category_id`, `second_category_id`, `third_category_id`, `product_id` | Identify cities, stores, products and the product-category hierarchy |
| Sales/Performance | `sale_amount`, `hours_sale` | Measure daily and hourly observed sales |
| Inventory | `stock_hour6_22_cnt`, `hours_stock_status` | Measure the number and timing of stockout hours |
| Promotion | `discount`, `activity_flag` | Describe discount intensity and promotional/activity periods |
| Calendar | `dt`, `holiday_flag` | Capture dates, time patterns and holidays |
| Weather | `precpt`, `avg_temperature`, `avg_humidity`, `avg_wind_level` | Describe external weather conditions |

The variables ending with `_id` are encoded identifiers rather than descriptive names. The category variables form a hierarchy from management group to product.

The `discount` variable is a discount multiplier. A value of `1.0` means no discount, while a value of `0.9` means a 10% discount.

## A3. Variables Relevant to the Management Decision

The selected Decision Track is **Inventory Optimisation**.

The most relevant variables are `sale_amount` and `stock_hour6_22_cnt`.

- `sale_amount` measures the observed sales performance of each product.
- `stock_hour6_22_cnt` measures the number of hours during which the product was out of stock.
- `city_id`, `store_id` and `product_id` identify where the inventory issue occurs and which product should receive attention.
- `discount`, `holiday_flag` and `activity_flag` provide business context that may explain changes in sales and stockout risk.

These variables can be used to identify products and stores with high observed sales but frequent stockouts. Such products may deserve additional inventory because their observed sales may be constrained by limited availability.

In [6]:
example_columns = [
    "city_id",
    "store_id",
    "product_id",
    "dt",
    "sale_amount",
    "stock_hour6_22_cnt",
    "discount",
    "holiday_flag",
    "activity_flag"
]

stockout_example = None

for batch in parquet_file.iter_batches(
    batch_size=100_000,
    columns=example_columns
):
    batch_df = batch.to_pandas()

    candidate = batch_df[
        (batch_df["stock_hour6_22_cnt"] > 0) &
        (batch_df["sale_amount"] == 0)
    ]

    if not candidate.empty:
        stockout_example = candidate.head(1)
        break

print(stockout_example)

    city_id  store_id  product_id          dt  sale_amount  \
44        0         0          38  2024-05-11          0.0   

    stock_hour6_22_cnt  discount  holiday_flag  activity_flag  
44                   2      0.75             1              1  


## A4. When Observed Sales Does Not Equal True Demand

Observed Sales may not equal True Demand when a product is out of stock.

For example, customers may want to purchase a product, but the product may be unavailable for several hours. In this situation, the recorded sales are limited by product availability rather than by the absence of customer demand.

Therefore, a low or zero `sale_amount` does not automatically mean that customer demand was low. The `stock_hour6_22_cnt` and `hours_stock_status` variables must also be considered when interpreting sales performance.

## A5. A Potentially Misleading Variable

The `sale_amount` variable can be misleading if it is interpreted as True Demand.

A low `sale_amount` may indicate low demand, but it may also result from stockouts, limited product availability, holidays, promotions or other contextual factors.

For this reason, `sale_amount` should be interpreted together with inventory, promotion, calendar and weather variables rather than being analyzed in isolation.

# Part B – Data Quality and Preparation

The original `train.parquet` file is kept unchanged. This section checks data quality, records any issues, and creates analysis variables in a separate prepared dataset.

In [7]:
import pandas as pd
import numpy as np

scalar_columns = [
    "city_id", "store_id", "management_group_id",
    "first_category_id", "second_category_id", "third_category_id",
    "product_id", "dt", "sale_amount", "stock_hour6_22_cnt",
    "discount", "holiday_flag", "activity_flag",
    "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"
]

missing_counts = pd.Series(0, index=scalar_columns, dtype="int64")
numeric_min = {}
numeric_max = {}
invalid_date_count = 0
empty_hours_sale = 0
empty_stock_status = 0

for batch in parquet_file.iter_batches(batch_size=100_000):
    part = batch.to_pandas()

    missing_counts += part[scalar_columns].isna().sum()

    dates = pd.to_datetime(part["dt"], errors="coerce")
    invalid_date_count += dates.isna().sum()

    for column in part.select_dtypes(include="number").columns:
        if part[column].notna().any():
            numeric_min[column] = min(
                numeric_min.get(column, float("inf")),
                part[column].min()
            )
            numeric_max[column] = max(
                numeric_max.get(column, float("-inf")),
                part[column].max()
            )

    empty_hours_sale += part["hours_sale"].map(
        lambda x: len(x) == 0 if x is not None else True
    ).sum()

    empty_stock_status += part["hours_stock_status"].map(
        lambda x: len(x) == 0 if x is not None else True
    ).sum()

print("Missing values by column:")
print(missing_counts[missing_counts > 0])

print("\nInvalid dates:", invalid_date_count)
print("Empty hours_sale lists:", empty_hours_sale)
print("Empty hours_stock_status lists:", empty_stock_status)

print("\nNumeric minimums:")
print(pd.Series(numeric_min).sort_index())

print("\nNumeric maximums:")
print(pd.Series(numeric_max).sort_index())

Missing values by column:
Series([], dtype: int64)

Invalid dates: 0
Empty hours_sale lists: 0
Empty hours_stock_status lists: 0

Numeric minimums:
activity_flag           0.00
avg_humidity           27.35
avg_temperature        12.06
avg_wind_level          0.97
city_id                 0.00
discount                0.00
first_category_id       0.00
holiday_flag            0.00
management_group_id     0.00
precpt                  0.00
product_id              0.00
sale_amount             0.00
second_category_id      0.00
stock_hour6_22_cnt      0.00
store_id                0.00
third_category_id       0.00
dtype: float64

Numeric maximums:
activity_flag            1.000
avg_humidity           100.000
avg_temperature         30.880
avg_wind_level           3.850
city_id                 17.000
discount                 1.088
first_category_id       31.000
holiday_flag             1.000
management_group_id      6.000
precpt                  42.500
product_id             864.000
sale_amount  

In [8]:
key_columns = ["city_id", "store_id", "product_id", "dt"]

keys = pq.read_table(
    DATA_PATH,
    columns=key_columns
).to_pandas()

duplicate_count = keys.duplicated().sum()

print("Duplicate product-store-date rows:", duplicate_count)

if duplicate_count:
    print(keys[keys.duplicated(keep=False)].head(10))

Duplicate product-store-date rows: 0


In [9]:
checks = {
    "negative_sales": ("sale_amount", lambda s: s < 0),
    "stockout_hours_out_of_range": (
        "stock_hour6_22_cnt",
        lambda s: (s < 0) | (s > 16)
    ),
    "discount_out_of_range": (
        "discount",
        lambda s: (s < 0) | (s > 1)
    ),
    "invalid_holiday_flag": ("holiday_flag", lambda s: ~s.isin([0, 1])),
    "invalid_activity_flag": ("activity_flag", lambda s: ~s.isin([0, 1])),
    "negative_precipitation": ("precpt", lambda s: s < 0),
    "humidity_out_of_range": (
        "avg_humidity",
        lambda s: (s < 0) | (s > 100)
    ),
    "negative_wind_level": ("avg_wind_level", lambda s: s < 0),
}

issue_counts = {name: 0 for name in checks}

for batch in parquet_file.iter_batches(
    batch_size=100_000,
    columns=list({column for column, _ in checks.values()})
):
    part = batch.to_pandas()

    for name, (column, condition) in checks.items():
        issue_counts[name] += condition(part[column]).fillna(False).sum()

print(pd.Series(issue_counts, name="rows_to_review"))

negative_sales                  0
stockout_hours_out_of_range     0
discount_out_of_range          34
invalid_holiday_flag            0
invalid_activity_flag           0
negative_precipitation          0
humidity_out_of_range           0
negative_wind_level             0
Name: rows_to_review, dtype: int64


In [10]:
wrong_length = 0
non_binary_values = 0
mismatch_6_22 = 0
rows_checked = 0

for batch in parquet_file.iter_batches(
    batch_size=50_000,
    columns=["stock_hour6_22_cnt", "hours_stock_status"]
):
    part = batch.to_pandas()

    statuses = part["hours_stock_status"]
    lengths = statuses.map(lambda x: len(x) if x is not None else -1)
    wrong_length += (lengths != 24).sum()

    valid = lengths == 24
    if not valid.any():
        continue

    status_matrix = np.stack(statuses[valid].map(np.asarray))
    counts = part.loc[valid, "stock_hour6_22_cnt"].to_numpy()

    non_binary_values += (~np.isin(status_matrix, [0, 1])).sum()

    # Dataset coding confirmed by cross-check: 1 = stockout, 0 = available.
    # Check the 16 hourly slots from 06:00 up to 22:00.
    expected_stockout_hours = (status_matrix[:, 6:22] == 1).sum(axis=1)

    mismatch_6_22 += (expected_stockout_hours != counts).sum()
    rows_checked += valid.sum()

print("Rows with status list length other than 24:", wrong_length)
print("Status values outside 0/1:", non_binary_values)
print("Rows checked against stockout-hour count:", rows_checked)
print("Count mismatches for hours 06:00–22:00:", mismatch_6_22)

Rows with status list length other than 24: 0
Status values outside 0/1: 0
Rows checked against stockout-hour count: 4500000
Count mismatches for hours 06:00–22:00: 0


In [11]:
continuous_columns = [
    "sale_amount",
    "precpt",
    "avg_temperature",
    "avg_humidity",
    "avg_wind_level"
]

outlier_results = []

for column in continuous_columns:
    values = (
        pq.read_table(DATA_PATH, columns=[column])
        .to_pandas()[column]
        .to_numpy()
    )

    q1, q3 = np.quantile(values, [0.25, 0.75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    below = int((values < lower_fence).sum())
    above = int((values > upper_fence).sum())

    outlier_results.append({
        "Variable": column,
        "Q1": q1,
        "Q3": q3,
        "LowerFence": lower_fence,
        "UpperFence": upper_fence,
        "BelowFence": below,
        "AboveFence": above,
        "TotalFlagged": below + above
    })

display(pd.DataFrame(outlier_results).round(3))

,Variable,Q1,Q3,LowerFence,UpperFence,BelowFence,AboveFence,TotalFlagged
0,sale_amount,0.400,1.100,-0.650,2.150,0,395451,395451
1,precpt,1.429,4.482,-3.150,9.061,0,441643,441643
2,avg_temperature,19.390,25.400,10.375,34.415,0,0,0
3,avg_humidity,71.620,80.710,57.985,94.345,245388,3,245391
4,avg_wind_level,1.450,1.950,0.700,2.700,0,69942,69942


In [12]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

prepared_path = Path("../data/processed/train_prepared.parquet")
prepared_path.parent.mkdir(parents=True, exist_ok=True)

writer = None

for batch in parquet_file.iter_batches(batch_size=50_000):
    part = batch.to_pandas()

    part["dt"] = pd.to_datetime(part["dt"], errors="coerce")

    stock_hours = part["stock_hour6_22_cnt"]
    part["HasStockout"] = (stock_hours > 0).where(stock_hours.notna())
    part["StockoutRate"] = stock_hours / 16
    valid_discount = part["discount"].between(0, 1)
    part["DiscountOutOfRange"] = ~valid_discount
    part["DiscountPercent"] = ((1 - part["discount"]) * 100).where(valid_discount)
    part["DayOfWeek"] = part["dt"].dt.day_name()
    part["WeekendFlag"] = part["dt"].dt.dayofweek.ge(5).astype("Int8")

    table = pa.Table.from_pandas(part, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(prepared_path, table.schema)

    writer.write_table(table)

if writer is not None:
    writer.close()

print("Prepared data saved to:", prepared_path)

Prepared data saved to: ..\data\processed\train_prepared.parquet


## Data Preparation Notes

| Check | Result | Action | Reason |
|---|---|---|---|
| Missing values | 0 missing values in the checked scalar columns; 0 empty `hours_sale` and `hours_stock_status` lists | Keep; no imputation | No missing or empty values were detected, so imputation is unnecessary. |
| Duplicate product-store-date keys | 0 duplicate rows | Keep | The key represents one product-store-date observation, and no duplicates were found. |
| Unusual values (IQR screening) | `sale_amount`: 395,451 values above the upper fence (2.150); `precpt`: 441,643 above (9.061); `avg_temperature`: 0; `avg_humidity`: 245,388 below the lower fence (57.985) and 3 above the upper fence (94.345); `avg_wind_level`: 69,942 above (2.700) | Flag for review; retain original values; do not automatically delete or cap them | The 1.5×IQR rule identifies statistical extremes, not necessarily errors. High sales, precipitation, or wind and low humidity may be valid observations. |
| Invalid dates | 0 invalid dates | Keep; no rows need to be excluded | Every date was successfully parsed. |
| Discount values | `discount` ranges from 0 to 1.088; 34 rows are outside the expected [0, 1] range | Retain the original `discount`; create `DiscountPercent` for valid values and flag out-of-range values | `discount` is a multiplier: 1.0 means no discount. Values above 1 produce negative `DiscountPercent` and need review before interpretation. |
| Weather values | No negative precipitation, humidity outside [0, 100], or negative wind values were detected. `avg_temperature` ranges from 12.06 to 30.88. IQR flags are summarized above. | Keep values within basic physical checks; flag statistical extremes for review | Statistical extremes are not necessarily physically impossible weather values. |
| Stock status consistency | All 4,500,000 rows have 24 binary status values; 0 mismatches between the number of 1s in slots [6:22] and `stock_hour6_22_cnt` | Keep; no mismatch correction needed | The hourly status values reconcile with the reported stockout-hour count for the 16 checked slots. |

### Assumptions

- The original `train.parquet` is preserved unchanged; engineered columns are saved in a separate prepared file.
- In this dataset, `hours_stock_status` uses 1 for stockout and 0 for available. This interpretation is based on its exact agreement with `stock_hour6_22_cnt`.
- `StockoutRate` uses 16 operating-hour slots, corresponding to indices [6:22].
- IQR fences are a screening rule applied to row-level values; flagged values are reviewed, not automatically treated as errors or deleted. Weather measurements may repeat across product-store-day records.
- Observed sales may be lower than true demand during stockout periods.

# Part C – KPI and Sales Performance

In [13]:
from collections import defaultdict
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

prepared_path = Path("../data/processed/train_prepared.parquet")
prepared_file = pq.ParquetFile(prepared_path)

columns = [
    "city_id", "store_id", "product_id", "dt", "sale_amount",
    "stock_hour6_22_cnt", "HasStockout", "DiscountPercent",
    "WeekendFlag", "holiday_flag", "activity_flag"
]

def new_totals():
    return defaultdict(lambda: [0.0, 0])

def add_group(totals, frame, by):
    grouped = frame.groupby(by)["sale_amount"].agg(["sum", "count"])
    for key, row in grouped.iterrows():
        if not isinstance(key, tuple):
            key = (key,)
        totals[key][0] += float(row["sum"])
        totals[key][1] += int(row["count"])

def to_table(totals, key_names):
    rows = [
        (*key, values[0], values[1], values[0] / values[1])
        for key, values in totals.items()
    ]
    return pd.DataFrame(
        rows,
        columns=key_names + ["TotalSales", "Records", "AvgSalesPerRecord"]
    )

group_columns = {
    "date": ["dt"],
    "product": ["product_id"],
    "store": ["city_id", "store_id"],
    "city": ["city_id"],
    "weekend": ["WeekendFlag"],
    "holiday": ["holiday_flag"],
    "activity": ["activity_flag"],
}
totals = {name: new_totals() for name in group_columns}

rows = 0
total_sales = 0.0
stockout_hours = 0
stockout_records = 0
discount_sum = 0.0
discount_count = 0
cities, stores, products = set(), set(), set()

for batch in prepared_file.iter_batches(batch_size=100_000, columns=columns):
    part = batch.to_pandas()

    rows += len(part)
    total_sales += part["sale_amount"].sum()
    stockout_hours += part["stock_hour6_22_cnt"].sum()
    stockout_records += int(part["HasStockout"].sum())
    discount_sum += part["DiscountPercent"].sum()
    discount_count += part["DiscountPercent"].count()

    cities.update(part["city_id"].unique())
    stores.update(zip(part["city_id"], part["store_id"]))
    products.update(part["product_id"].unique())

    for name, group_cols in group_columns.items():
        add_group(totals[name], part, group_cols)

daily = to_table(totals["date"], ["dt"])
product_sales = to_table(totals["product"], ["product_id"])
store_sales = to_table(totals["store"], ["city_id", "store_id"])
city_sales = to_table(totals["city"], ["city_id"])

print("Total sales:", total_sales)
print("Average total sales per day:", daily["TotalSales"].mean())
print("Number of stores:", len(stores))
print("Number of products:", len(products))
print("Number of cities:", len(cities))
print("Stockout-hour rate (%):", stockout_hours / (rows * 16) * 100)
print("Product-store-day records with any stockout (%):",
      stockout_records / rows * 100)
print("Average discount (%):", discount_sum / discount_count)

print("\nTop 10 products by total sales:")
display(product_sales.sort_values("TotalSales", ascending=False).head(10))

print("\nCity sales:")
display(city_sales.sort_values("TotalSales", ascending=False))

print("\nTop 10 stores by total sales:")
display(store_sales.sort_values("TotalSales", ascending=False).head(10))

for name, keys, labels in [
    ("weekend", ["WeekendFlag"], {0: "Weekday", 1: "Weekend"}),
    ("holiday", ["holiday_flag"], {0: "Non-holiday", 1: "Holiday"}),
    ("activity", ["activity_flag"], {0: "Normal", 1: "Activity/Promotion"}),
]:
    table = to_table(totals[name], keys)
    table["Group"] = table[keys[0]].map(labels)
    print(f"\nSales comparison: {name}")
    display(table[["Group", "TotalSales", "Records", "AvgSalesPerRecord"]])

Total sales: 4493660.7870000005
Average total sales per day: 49929.5643
Number of stores: 898
Number of products: 865
Number of cities: 18
Stockout-hour rate (%): 19.877133333333333
Product-store-day records with any stockout (%): 44.2668
Average discount (%): 8.885996559690406

Top 10 products by total sales:


,product_id,TotalSales,Records,AvgSalesPerRecord
688,267,183395.100,11790,15.555140
1,4,183265.300,70470,2.600614
114,300,154505.577,37080,4.166817
55,117,150032.945,41310,3.631880
86,215,144031.118,55260,2.606426
31,70,120042.148,33390,3.595153
182,580,117250.900,66150,1.772500
264,834,87215.500,66690,1.307775
220,691,66486.590,17280,3.847604
583,191,66122.317,32130,2.057962



City sales:


,city_id,TotalSales,Records,AvgSalesPerRecord
0,0,2374623.838,2322990,1.022227
12,12,429614.580,460530,0.932870
16,16,402280.885,407520,0.987144
6,6,221633.112,214560,1.032966
13,13,209188.069,210240,0.994997
11,11,190952.319,168300,1.134595
3,3,161878.525,238320,0.679249
4,4,136821.517,102690,1.332374
15,15,68081.183,72270,0.942039
5,5,53397.982,60930,0.876382



Top 10 stores by total sales:


...

,Group,TotalSales,Records,AvgSalesPerRecord
0,Weekday,2947424.998,3200000,0.921070
1,Weekend,1546235.789,1300000,1.189412



Sales comparison: holiday


,Group,TotalSales,Records,AvgSalesPerRecord
0,Non-holiday,2696208.536,2950000,0.913969
1,Holiday,1797452.251,1550000,1.159647



Sales comparison: activity


,Group,TotalSales,Records,AvgSalesPerRecord
0,Normal,2768190.609,2797107,0.989662
1,Activity/Promotion,1725470.178,1702893,1.013258


## Findings

### Finding 1: Top-selling products
- **Observation:** Product 267 had the highest total sales at 183,395.1 across 11,790 product-store-day records. Product 4 was nearly tied at 183,265.3, but across 70,470 records. Product 267's average sales per record were 15.56, compared with 2.60 for Product 4.
- **Insight:** Product 267 may deserve closer inventory review, but the large difference in record counts means total sales alone may be misleading. Check product coverage and stockout patterns before making replenishment decisions.

### Finding 2: Weekday vs. weekend
- **Observation:** Average sales per product-store-day were 0.921 on weekdays and 1.189 on weekends, about 29.1% higher on weekends.
- **Insight:** This pattern may help with weekend replenishment and staffing plans. It is an association, not proof that weekends caused the increase.

### Finding 3: Holiday vs. non-holiday
- **Observation:** Average sales per product-store-day were 0.914 on non-holidays and 1.160 on holidays, about 26.9% higher on holidays.
- **Insight:** Holiday periods may need closer inventory planning. The difference could also reflect promotions, stockouts, or which stores and products were observed.

### Finding 4: Activity/promotion vs. normal periods
- **Observation:** Average sales per product-store-day were 0.990 during normal periods and 1.013 when `activity_flag` was 1, about 2.4% higher.
- **Insight:** The observed difference is modest. Do not conclude that the activity caused higher sales without controlling for discounts, holidays, and stock availability.

### Finding 5: City and store sales
- **Observation:** City 0 had the highest total sales at 2,374,623.8 across 2,322,990 records. The highest-sales store in the results was Store 235 in City 0, with 17,510.5 across 14,580 records.
- **Insight:** City 0 and Store 235 are candidates for closer review, but total sales also depend on the number of records. Compare average sales per record and stockout rates before allocating more inventory.

# Part D – Stockout Risk and Product Prioritisation

In [14]:
from collections import defaultdict
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

prepared_path = Path("../data/processed/train_prepared.parquet")
prepared_file = pq.ParquetFile(prepared_path)

columns = [
    "city_id", "store_id", "product_id", "sale_amount",
    "stock_hour6_22_cnt", "HasStockout", "DiscountPercent",
    "holiday_flag", "activity_flag"
]

def new_summary():
    return defaultdict(lambda: [0, 0.0, 0, 0.0])

def add_summary(target, frame, keys):
    grouped = frame.groupby(keys).agg(
        Records=("sale_amount", "size"),
        TotalSales=("sale_amount", "sum"),
        StockoutDays=("HasStockout", "sum"),
        StockoutHours=("stock_hour6_22_cnt", "sum")
    )

    for key, row in grouped.iterrows():
        if not isinstance(key, tuple):
            key = (key,)

        values = target[key]
        values[0] += int(row["Records"])
        values[1] += float(row["TotalSales"])
        values[2] += int(row["StockoutDays"])
        values[3] += float(row["StockoutHours"])

def make_summary(target, key_names):
    rows = []

    for key, values in target.items():
        records, sales, stockout_days, stockout_hours = values
        rows.append((
            *key,
            records,
            sales / records,
            100 * stockout_days / records,
            stockout_hours / stockout_days if stockout_days else 0
        ))

    return pd.DataFrame(
        rows,
        columns=key_names + [
            "Records", "AvgSales", "StockoutRatePct",
            "AvgHoursOnStockoutDays"
        ]
    )

product_totals = new_summary()
store_totals = new_summary()
stockout_comparison = {
    False: defaultdict(float),
    True: defaultdict(float)
}

total_records = 0
total_stockout_hours = 0

for batch in prepared_file.iter_batches(
    batch_size=100_000,
    columns=columns
):
    part = batch.to_pandas()
    part["HasStockout"] = part["HasStockout"].astype(bool)

    total_records += len(part)
    total_stockout_hours += part["stock_hour6_22_cnt"].sum()

    add_summary(product_totals, part, ["product_id"])
    add_summary(store_totals, part, ["city_id", "store_id"])

    for has_stockout, group in part.groupby("HasStockout"):
        values = stockout_comparison[bool(has_stockout)]
        values["Records"] += len(group)
        values["Sales"] += group["sale_amount"].sum()
        values["DiscountSum"] += group["DiscountPercent"].sum()
        values["DiscountCount"] += group["DiscountPercent"].count()
        values["HolidaySum"] += group["holiday_flag"].sum()
        values["ActivitySum"] += group["activity_flag"].sum()

product_summary = make_summary(product_totals, ["product_id"])
store_summary = make_summary(store_totals, ["city_id", "store_id"])

stockout_rows = stockout_comparison[True]["Records"]
stockout_frequency = 100 * stockout_rows / total_records
avg_stockout_duration = total_stockout_hours / stockout_rows

print("Records with any stockout (%):", stockout_frequency)
print("Average duration on stockout records (hours):", avg_stockout_duration)

comparison_rows = []
for has_stockout, label in [(False, "No stockout"), (True, "Stockout")]:
    values = stockout_comparison[has_stockout]
    n = values["Records"]

    comparison_rows.append({
        "Group": label,
        "Records": int(n),
        "AvgSales": values["Sales"] / n,
        "AvgDiscountPercent": (
            values["DiscountSum"] / values["DiscountCount"]
        ),
        "HolidayRecordsPct": 100 * values["HolidaySum"] / n,
        "ActivityRecordsPct": 100 * values["ActivitySum"] / n
    })

display(pd.DataFrame(comparison_rows).round(3))

Records with any stockout (%): 44.2668
Average duration on stockout records (hours): 7.184484384083181


,Group,Records,AvgSales,AvgDiscountPercent,HolidayRecordsPct,ActivityRecordsPct
0,No stockout,2507994,0.975,8.455,34.340,36.329
1,Stockout,1992006,1.029,9.428,34.576,39.747


In [15]:
sales_threshold = product_summary["AvgSales"].median()
stockout_threshold = product_summary["StockoutRatePct"].median()

product_summary["SalesLevel"] = product_summary["AvgSales"].ge(
    sales_threshold
).map({True: "High Sales", False: "Low Sales"})

product_summary["StockoutLevel"] = product_summary["StockoutRatePct"].ge(
    stockout_threshold
).map({True: "High Stockout", False: "Low Stockout"})

matrix = pd.crosstab(
    product_summary["SalesLevel"],
    product_summary["StockoutLevel"]
).reindex(
    index=["High Sales", "Low Sales"],
    columns=["High Stockout", "Low Stockout"],
    fill_value=0
)

print("Median sales threshold:", sales_threshold)
print("Median stockout-rate threshold (%):", stockout_threshold)
print("2x2 product matrix:")
display(matrix)

# Shortlist products with enough observations to make the rate more stable
high_high_products = product_summary[
    (product_summary["SalesLevel"] == "High Sales") &
    (product_summary["StockoutLevel"] == "High Stockout") &
    (product_summary["Records"] >= 1000)
].sort_values(
    ["StockoutRatePct", "AvgSales"],
    ascending=False
)

print("High Sales + High Stockout products:")
display(high_high_products[
    [
        "product_id", "AvgSales", "StockoutRatePct",
        "AvgHoursOnStockoutDays", "Records"
    ]
].head(10).round(3))

print("Products with highest stockout rate, with at least 1,000 records:")
display(product_summary[
    product_summary["Records"] >= 1000
].sort_values(
    "StockoutRatePct",
    ascending=False
)[
    [
        "product_id", "AvgSales", "StockoutRatePct",
        "AvgHoursOnStockoutDays", "Records"
    ]
].head(10).round(3))

print("Stores with highest stockout rate, with at least 500 records:")
display(store_summary[
    store_summary["Records"] >= 500
].sort_values(
    "StockoutRatePct",
    ascending=False
)[
    [
        "city_id", "store_id", "AvgSales", "StockoutRatePct",
        "AvgHoursOnStockoutDays", "Records"
    ]
].head(10).round(3))

Median sales threshold: 0.5214506172839506
Median stockout-rate threshold (%): 37.77777777777778
2x2 product matrix:


StockoutLevel,High Stockout,Low Stockout
SalesLevel,,
High Sales,203,230
Low Sales,230,202


High Sales + High Stockout products:


,product_id,AvgSales,StockoutRatePct,AvgHoursOnStockoutDays,Records
93,224,0.753,99.228,5.341,5310
94,228,0.721,97.946,5.717,2970
277,226,0.798,97.604,4.966,2880
333,313,0.553,96.790,5.980,2430
542,589,10.793,81.178,5.443,4500
134,379,0.681,80.896,9.172,10710
571,157,1.046,79.929,8.347,5670
688,267,15.555,78.134,7.579,11790
34,74,0.562,76.879,5.786,16020
186,600,0.842,75.111,7.605,44730


Products with highest stockout rate, with at least 1,000 records:


,product_id,AvgSales,StockoutRatePct,AvgHoursOnStockoutDays,Records
93,224,0.753,99.228,5.341,5310
94,228,0.721,97.946,5.717,2970
277,226,0.798,97.604,4.966,2880
333,313,0.553,96.790,5.980,2430
281,344,0.509,91.265,5.804,5850
254,816,0.489,82.386,4.397,3060
542,589,10.793,81.178,5.443,4500
134,379,0.681,80.896,9.172,10710
224,706,0.471,80.463,4.563,1080
571,157,1.046,79.929,8.347,5670


Stores with highest stockout rate, with at least 500 records:


,city_id,store_id,AvgSales,StockoutRatePct,AvgHoursOnStockoutDays,Records
531,9,36,1.037,57.515,6.245,3420
312,2,97,0.895,56.524,5.531,3510
533,9,38,0.835,55.767,6.938,1890
730,13,673,0.758,55.714,6.983,1890
522,7,859,0.982,55.625,5.707,1440
348,3,725,0.564,54.444,9.219,630
774,13,810,1.356,54.183,6.964,1530
301,1,123,0.751,54.167,7.192,1800
773,13,798,0.803,54.061,6.682,2610
307,2,45,1.173,53.983,5.926,8460


In [16]:
from collections import defaultdict
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

prepared_file = pq.ParquetFile(
    Path("../data/processed/train_prepared.parquet")
)

pair_totals = defaultdict(lambda: [0, 0.0, 0, 0.0])

for batch in prepared_file.iter_batches(
    batch_size=100_000,
    columns=[
        "product_id", "city_id", "store_id",
        "sale_amount", "HasStockout", "stock_hour6_22_cnt"
    ]
):
    part = batch.to_pandas()

    grouped = part.groupby(
        ["product_id", "city_id", "store_id"]
    ).agg(
        Records=("sale_amount", "size"),
        TotalSales=("sale_amount", "sum"),
        StockoutRecords=("HasStockout", "sum"),
        StockoutHours=("stock_hour6_22_cnt", "sum")
    )

    for (product, city, store), row in grouped.iterrows():
        values = pair_totals[(int(product), int(city), int(store))]
        values[0] += int(row["Records"])
        values[1] += float(row["TotalSales"])
        values[2] += int(row["StockoutRecords"])
        values[3] += float(row["StockoutHours"])

pair_summary = pd.DataFrame(
    [
        (
            product, city, store,
            records,
            sales / records,
            100 * stockout_records / records,
            stockout_hours / stockout_records
            if stockout_records else 0
        )
        for (product, city, store),
            (records, sales, stockout_records, stockout_hours)
        in pair_totals.items()
    ],
    columns=[
        "product_id", "city_id", "store_id", "Records",
        "AvgSales", "StockoutRatePct", "AvgStockoutHours"
    ]
)

# Chỉ xếp hạng các tổ hợp có ít nhất 60 ngày quan sát
eligible_pairs = pair_summary[pair_summary["Records"] >= 60].copy()

sales_cutoff = eligible_pairs["AvgSales"].median()
stockout_cutoff = eligible_pairs["StockoutRatePct"].median()

eligible_pairs["SalesLevel"] = eligible_pairs["AvgSales"].ge(
    sales_cutoff
).map({True: "High Sales", False: "Low Sales"})

eligible_pairs["StockoutLevel"] = eligible_pairs["StockoutRatePct"].ge(
    stockout_cutoff
).map({True: "High Stockout", False: "Low Stockout"})

print("Eligible product-store pairs:", len(eligible_pairs))
print("Minimum records per pair: 60")
print("Pair-level median sales threshold:", sales_cutoff)
print("Pair-level median stockout threshold (%):", stockout_cutoff)

pair_matrix = pd.crosstab(
    eligible_pairs["SalesLevel"],
    eligible_pairs["StockoutLevel"]
).reindex(
    index=["High Sales", "Low Sales"],
    columns=["High Stockout", "Low Stockout"],
    fill_value=0
)
display(pair_matrix)

priority_pairs = eligible_pairs[
    (eligible_pairs["SalesLevel"] == "High Sales") &
    (eligible_pairs["StockoutLevel"] == "High Stockout")
].sort_values(
    ["StockoutRatePct", "AvgSales", "Records"],
    ascending=[False, False, False]
)

display(
    priority_pairs[
        [
            "product_id", "city_id", "store_id", "Records",
            "AvgSales", "StockoutRatePct", "AvgStockoutHours"
        ]
    ].head(10).round(3)
)

Eligible product-store pairs: 50000
Minimum records per pair: 60
Pair-level median sales threshold: 0.6577777777777778
Pair-level median stockout threshold (%): 42.22222222222222


StockoutLevel,High Stockout,Low Stockout
SalesLevel,,
High Sales,14132,10871
Low Sales,11605,13392


,product_id,city_id,store_id,Records,AvgSales,StockoutRatePct,AvgStockoutHours
29320,267,4,11,90,17.891,100.0,8.211
34816,267,11,505,90,16.652,100.0,8.067
34768,224,11,602,90,1.304,100.0,4.156
19335,228,0,551,90,1.163,100.0,5.067
31499,226,6,362,90,1.163,100.0,4.111
32633,227,6,537,90,1.134,100.0,4.256
31501,226,6,469,90,1.117,100.0,4.556
32627,226,6,537,90,1.087,100.0,5.044
38141,224,12,261,90,1.078,100.0,6.256
43685,228,14,77,90,1.066,100.0,5.533


## Findings

### Finding 1: Stockout frequency and duration
- **Observation:** 44.27% of product-store-day records had at least one stockout. Among records with a stockout, the average duration was 7.18 hours.
- **Insight:** Stockouts occur frequently and last several hours on average. Replenishment timing and inventory availability should be reviewed.

### Finding 2: High-sales, high-stockout products
- **Observation:** The median thresholds were 0.521 average sales and 37.78% stockout frequency. The 2×2 matrix placed 203 products in the High Sales + High Stockout group. After requiring at least 1,000 records, Products 224, 228, and 226 were among the highest-priority examples, with stockout rates of 99.23%, 97.95%, and 97.60%.
- **Insight:** These products combine above-median sales with frequent stockouts and merit review for replenishment or safety-stock decisions. Their observed sales may understate demand during periods when they are unavailable.

### Finding 3: Stockout vs. no-stockout records
- **Observation:** Average observed sales were 1.029 on stockout records and 0.975 on records without stockouts. Stockout records also had a higher average discount (9.43% vs. 8.46%), activity rate (39.75% vs. 36.33%), and holiday-record share (34.58% vs. 34.34%).
- **Insight:** Stockouts coincide with somewhat higher sales, discounts, and activity in this dataset. These are associations, not proof that promotions or holidays caused stockouts. Observed sales alone cannot measure the exact demand that went unmet.

### Finding 4: Stores with frequent stockouts
- **Observation:** Among stores with at least 500 records, City 9, Store 36 had the highest stockout frequency at 57.52%, with an average stockout duration of 6.25 hours on stockout records. City 2, Store 97 followed at 56.52%.
- **Insight:** These stores are candidates for an inventory-process review. Check replenishment timing and local demand before concluding that either store was understocked.

# Part E – Discount and External Context

### Discount band rationale

The discount bands are analyst-defined for interpretability:
- No discount: exactly 0%.
- Low: greater than 0% and up to 10%.
- Medium: greater than 10% and up to 30%.
- High: greater than 30% and up to 100%.

Zero is separated because it represents no price reduction. The 10% and 30% cutoffs provide simple, increasing levels of discount intensity that are easy to compare in the dashboard. These are practical analysis thresholds, not universal business rules or statistically optimized cutoffs. The 34 records outside the valid discount range are excluded from these comparisons and retained in the source data.

In [17]:
from collections import defaultdict
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

prepared_file = pq.ParquetFile(
    Path("../data/processed/train_prepared.parquet")
)

columns = [
    "city_id", "sale_amount", "HasStockout",
    "DiscountPercent", "DiscountOutOfRange", "activity_flag"
]

band_labels = [
    "No discount",
    "Low (0-10%)",
    "Medium (10-30%)",
    "High (>30%)"
]

discount_totals = defaultdict(lambda: [0, 0.0, 0])
city_activity_totals = defaultdict(lambda: [0, 0.0])
invalid_discount_rows = 0

for batch in prepared_file.iter_batches(
    batch_size=100_000,
    columns=columns
):
    part = batch.to_pandas()
    invalid_discount_rows += int(part["DiscountOutOfRange"].sum())

    part["DiscountBand"] = pd.cut(
        part["DiscountPercent"],
        bins=[-0.001, 0, 10, 30, float("inf")],
        labels=band_labels,
        include_lowest=True
    )

    valid_discount = part.dropna(subset=["DiscountBand"])
    for band, group in valid_discount.groupby("DiscountBand", observed=True):
        values = discount_totals[str(band)]
        values[0] += len(group)
        values[1] += group["sale_amount"].sum()
        values[2] += int(group["HasStockout"].sum())

    for (city, activity), group in part.groupby(
        ["city_id", "activity_flag"]
    ):
        values = city_activity_totals[(int(city), int(activity))]
        values[0] += len(group)
        values[1] += group["sale_amount"].sum()

discount_rows = []
for band in band_labels:
    records, sales, stockout_records = discount_totals[band]
    discount_rows.append({
        "DiscountBand": band,
        "Observations": records,
        "AvgSales": sales / records,
        "StockoutRatePct": 100 * stockout_records / records
    })

discount_summary = pd.DataFrame(discount_rows)
print("Invalid discount rows excluded:", invalid_discount_rows)
display(discount_summary.round(3))

no_discount_rate = discount_summary.loc[
    discount_summary["DiscountBand"] == "No discount",
    "StockoutRatePct"
].iloc[0]
high_discount_rate = discount_summary.loc[
    discount_summary["DiscountBand"] == "High (>30%)",
    "StockoutRatePct"
].iloc[0]

print(
    "High-discount minus no-discount stockout rate "
    "(percentage points):",
    round(high_discount_rate - no_discount_rate, 3)
)

activity_rows = []
for (city, activity), (records, sales) in city_activity_totals.items():
    activity_rows.append({
        "city_id": city,
        "Activity": "Activity" if activity == 1 else "Normal",
        "Observations": records,
        "AvgSales": sales / records
    })

activity_by_city = pd.DataFrame(activity_rows)
city_sales = activity_by_city.pivot(
    index="city_id",
    columns="Activity",
    values="AvgSales"
)
city_sales["ActivityChangePct"] = (
    city_sales["Activity"] / city_sales["Normal"] - 1
) * 100

print(
    "Cities with higher average sales during activity:",
    int((city_sales["ActivityChangePct"] > 0).sum()),
    "of",
    len(city_sales)
)
display(city_sales.round(3))

Invalid discount rows excluded: 34


,DiscountBand,Observations,AvgSales,StockoutRatePct
0,No discount,2181121,0.833,43.030
1,Low (0-10%),724658,1.024,43.525
2,Medium (10-30%),1275243,1.001,46.158
3,High (>30%),318944,2.064,46.850


High-discount minus no-discount stockout rate (percentage points): 3.821
Cities with higher average sales during activity: 9 of 18


Activity,Activity,Normal,ActivityChangePct
city_id,,,
0,1.054,1.001,5.308
1,0.941,1.051,-10.446
2,0.999,0.948,5.340
3,0.702,0.668,5.110
4,0.978,1.547,-36.783
5,0.883,0.873,1.176
6,1.024,1.037,-1.316
7,0.831,0.997,-16.652
8,0.704,0.813,-13.399


## Findings

### Finding 1: Discount groups and sales
- **Observation:** Average sales per record were 0.833 for no discount (2,181,121 observations), 1.024 for low discount (724,658), 1.001 for medium discount (1,275,243), and 2.064 for high discount (318,944).
- **Insight:** The high-discount group had the highest observed average sales, but sales did not rise steadily across all discount bands. Product mix, activity, holidays, and stock availability may also contribute.

### Finding 2: Discount groups and stockout risk
- **Observation:** Stockout rates were 43.03% with no discount, 43.53% with low discount, 46.16% with medium discount, and 46.85% with high discount. The high-discount rate was 3.82 percentage points above the no-discount rate.
- **Insight:** Higher discount bands were associated with higher stockout rates in this dataset. This does not prove that discounts caused stockouts; discounts may coincide with higher-demand products or periods.

### Finding 3: Activity and consistency across cities
- **Observation:** Average sales during activity periods were higher in 9 of 18 cities and lower in the other 9. The difference ranged from about +18.34% in City 12 to -36.78% in City 4.
- **Insight:** The activity–sales relationship was not consistent across cities. A single overall average may hide local differences, so activity results should be checked by city before making a broad promotion decision.

### Assumptions and limitations
- Discount bands are analyst-defined: no discount = 0%; low = >0–10%; medium = >10–30%; high = >30%.
- The 34 out-of-range discount records are excluded from discount-band comparisons and remain unchanged in the source data.
- `activity_flag` indicates an activity period; it does not isolate the effect of a discount.
- These are observational comparisons. Correlation does not establish causation.

# Part G – Decision Track: Inventory Optimisation

## Selected Track: Track A

**Decision question:** Which products and stores should receive additional inventory?

### Evidence from the analysis
- 44.27% of product-store-day records had at least one stockout. On stockout records, the average stockout duration was 7.18 hours.
- Products 224, 228, and 226 were among the high-sales, high-stockout products. Their stockout rates were 99.23%, 97.95%, and 97.60%, respectively.
- The store-level analysis identified City 9, Store 36 (57.52% stockout rate) and City 2, Store 97 (56.52%) as stores for review.
- The pair-level analysis included 50,000 product-store combinations with at least 60 observations. Using pair-level medians (average sales = 0.658; stockout rate = 42.22%), 14,132 pairs were classified as High Sales + High Stockout.
- Leading candidates were Product 267, City 4, Store 11 (90 records; average sales 17.891; stockout rate 100%; average 8.211 stockout hours on stockout records), Product 267, City 11, Store 505 (90; 16.652; 100%; 8.067 hours), and Product 224, City 11, Store 602 (90; 1.304; 100%; 4.156 hours).
- The two Product 267 averages are unusually high relative to the row-level IQR upper fence for `sale_amount` (2.150). Keep them flagged for business validation before using them to set inventory; do not treat the statistical flag alone as proof of error.

### Recommendation
Run one monitored replenishment pilot, beginning with Product 224 at City 11, Store 602 after checking current inventory and supplier lead time. Validate the two Product 267 combinations against source sales and business context before including them in the pilot. The median cutoffs are relative ranking thresholds, not optimal reorder points.

Measure the result by comparing stockout frequency and observed sales before and after the change. Do not assume that a high stockout rate alone proves the store needs more inventory; observed sales can be constrained by stockouts, and the analysis does not measure exact lost demand.

# Part H – Unique Analytical Question

**Question:** Are weekend inventory needs consistent across cities, considering both average observed sales and stockout rates?

In [18]:
from collections import defaultdict
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

prepared_file = pq.ParquetFile(
    Path("../data/processed/train_prepared.parquet")
)

totals = defaultdict(lambda: [0, 0.0, 0])

for batch in prepared_file.iter_batches(
    batch_size=100_000,
    columns=["city_id", "WeekendFlag", "sale_amount", "HasStockout"]
):
    part = batch.to_pandas()

    grouped = part.groupby(["city_id", "WeekendFlag"]).agg(
        Records=("sale_amount", "size"),
        TotalSales=("sale_amount", "sum"),
        StockoutRecords=("HasStockout", "sum")
    )

    for (city, weekend), row in grouped.iterrows():
        values = totals[(int(city), int(weekend))]
        values[0] += int(row["Records"])
        values[1] += float(row["TotalSales"])
        values[2] += int(row["StockoutRecords"])

rows = []
for (city, weekend), (records, sales, stockouts) in totals.items():
    rows.append({
        "city_id": city,
        "DayType": "Weekend" if weekend == 1 else "Weekday",
        "Records": records,
        "AvgSales": sales / records,
        "StockoutRatePct": 100 * stockouts / records
    })

summary = pd.DataFrame(rows)

sales = summary.pivot(
    index="city_id", columns="DayType", values="AvgSales"
).reindex(columns=["Weekday", "Weekend"])

stockouts = summary.pivot(
    index="city_id", columns="DayType", values="StockoutRatePct"
).reindex(columns=["Weekday", "Weekend"])

records = summary.pivot(
    index="city_id", columns="DayType", values="Records"
).reindex(columns=["Weekday", "Weekend"])

city_weekend = pd.DataFrame(index=sales.index)
city_weekend["WeekdayAvgSales"] = sales["Weekday"]
city_weekend["WeekendAvgSales"] = sales["Weekend"]
city_weekend["SalesChangePct"] = (
    100 * (sales["Weekend"] / sales["Weekday"] - 1)
).where(sales["Weekday"] != 0)

city_weekend["WeekdayStockoutPct"] = stockouts["Weekday"]
city_weekend["WeekendStockoutPct"] = stockouts["Weekend"]
city_weekend["StockoutChangePP"] = (
    stockouts["Weekend"] - stockouts["Weekday"]
)

city_weekend["WeekdayRecords"] = records["Weekday"]
city_weekend["WeekendRecords"] = records["Weekend"]

print(
    "Cities with higher weekend average sales:",
    int((city_weekend["SalesChangePct"] > 0).sum()),
    "of", len(city_weekend)
)
print(
    "Cities with higher weekend stockout rate:",
    int((city_weekend["StockoutChangePP"] > 0).sum()),
    "of", len(city_weekend)
)

display(
    city_weekend.sort_values(
        "StockoutChangePP", ascending=False
    ).round(2)
)

Cities with higher weekend average sales: 18 of 18
Cities with higher weekend stockout rate: 11 of 18


,WeekdayAvgSales,WeekendAvgSales,SalesChangePct,WeekdayStockoutPct,WeekendStockoutPct,StockoutChangePP,WeekdayRecords,WeekendRecords
city_id,,,,,,,,
7,0.86,1.18,37.59,44.36,52.17,7.80,28416,11544
13,0.89,1.25,40.39,45.05,52.52,7.47,149504,60736
1,0.92,1.23,33.16,46.33,52.67,6.33,28864,11726
2,0.87,1.20,37.57,48.68,54.26,5.58,26048,10582
9,0.85,1.15,36.42,50.53,53.53,3.00,12608,5122
0,0.95,1.20,26.94,43.24,45.99,2.75,1651904,671086
12,0.85,1.14,34.51,44.17,46.27,2.10,327488,133042
3,0.62,0.83,35.25,40.61,42.05,1.44,169472,68848
8,0.72,0.97,35.53,37.40,37.99,0.58,4160,1690


### Answer

- **Observation:** Average observed sales were higher on weekends in all 18 cities, increasing by 23.76% to 40.39% compared with weekdays. However, weekend stockout rates were higher in 11 cities and lower in 7. The largest increases were in City 7 (+7.80 percentage points) and City 13 (+7.47 percentage points).
- **Insight:** Weekend sales are consistently higher across cities, but weekend inventory risk is not. City 7 and City 13 are candidates for closer weekend replenishment review; City 13 also has substantially more observations, while City 7 has the largest stockout-rate increase.
- **Limitation:** These are descriptive comparisons, not proof that weekends caused higher sales or stockouts. Holidays, activities, product mix, and unequal observation counts across cities may also affect the results. Observed sales may be constrained by stockouts.

# Part I: Management Recommendation and Limitations

In [19]:
import plotly.express as px

# Evidence 1: sản phẩm có doanh số và rủi ro hết hàng cao
product_fig = px.scatter(
    product_summary,
    x="AvgSales",
    y="StockoutRatePct",
    size="Records",
    hover_name="product_id",
    hover_data=["AvgHoursOnStockoutDays"],
    labels={
        "AvgSales": "Average observed sales per record",
        "StockoutRatePct": "Records with stockout (%)",
        "product_id": "Product ID"
    },
    title="Product sales vs. stockout risk"
)
product_fig.add_vline(
    x=product_summary["AvgSales"].median(),
    line_dash="dash"
)
product_fig.add_hline(
    y=product_summary["StockoutRatePct"].median(),
    line_dash="dash"
)
product_fig.show()

# Evidence 2: thay đổi tỷ lệ hết hàng cuối tuần theo thành phố
weekend_plot = city_weekend.reset_index()

weekend_fig = px.bar(
    weekend_plot,
    x="city_id",
    y="StockoutChangePP",
    color="StockoutChangePP",
    color_continuous_midpoint=0,
    hover_data=[
        "SalesChangePct", "WeekdayRecords", "WeekendRecords"
    ],
    labels={
        "city_id": "City ID",
        "StockoutChangePP": "Weekend minus weekday stockout rate (pp)"
    },
    title="Weekend change in stockout rate by city"
)
weekend_fig.add_hline(y=0, line_dash="dash")
weekend_fig.show()

# Part I – Management Recommendation and Limitations

## Business question
Which product–store combinations should receive priority for a targeted replenishment pilot next month?

## Evidence and relevant KPIs
- 44.27% of product-store-day records had at least one stockout.
- Average stockout duration was 7.18 hours on records with a stockout.
- Product 224 at City 11, Store 602 had a 100% stockout rate across 90 observations and average observed sales of 1.304. The two leading Product 267 pairs also had 100% stockout rates, but their average sales (17.891 and 16.652) require validation because they are unusually high.
- Weekend stockout rates were higher in 11 of 18 cities; the largest increases were City 7 (+7.80 percentage points) and City 13 (+7.47 percentage points).

## Visual evidence
1. The product sales vs. stockout-risk scatter plot highlights products with both above-median sales and stockout rates.
2. The weekend stockout-change chart shows that weekend risk varies by city; City 7 and City 13 had the largest increases.

## Observation
The overall stockout frequency was 44.27%, and stockouts lasted 7.18 hours on average when present. Product 224 at City 11, Store 602 had a 100% pair-level stockout rate across 90 observations. Weekend stockout rates rose in 11 cities but fell in 7; Cities 7 and 13 had the largest increases.

## Insight
Availability may be constraining observed sales for some product-store pairs, so replenishment should be targeted rather than increased everywhere. The city-level weekend pattern is not uniform and does not, by itself, prove that a specific product-store pair needs extra weekend stock.

## Decision
Fund one monitored replenishment pilot next month, starting with Product 224 at City 11, Store 602 after confirming current inventory and supplier lead time. Validate Product 267 at City 4, Store 11 and City 11, Store 505 before adding them, because their sales values are unusually high. Compare stockout rate, stockout duration, and observed sales before and during the pilot; use weekend checks only where the selected pairs support them.

## Limitation
These are observational data. Stockouts can censor observed sales, and the analysis does not estimate exact lost demand or the quantity of additional inventory required. The pair ranking uses a 60-observation minimum and relative median cutoffs, not an optimal stocking rule. The analysis also lacks inventory cost, supplier lead time, and profit margin; high stockout rates alone do not prove that additional stock is economically optimal.

# Optional Distinction Challenge – Product Segmentation

In [20]:
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

prepared_file = pq.ParquetFile(
    Path("../data/processed/train_prepared.parquet")
)

product_totals = defaultdict(lambda: [0, 0.0, 0.0, 0, 0.0, 0])

for batch in prepared_file.iter_batches(
    batch_size=100_000,
    columns=[
        "product_id", "sale_amount", "HasStockout", "DiscountPercent"
    ]
):
    part = batch.to_pandas()

    grouped = part.groupby("product_id").agg(
        Records=("sale_amount", "size"),
        Sales=("sale_amount", "sum"),
        SalesSquared=("sale_amount", lambda s: (s ** 2).sum()),
        StockoutRecords=("HasStockout", "sum"),
        DiscountSum=("DiscountPercent", "sum"),
        DiscountCount=("DiscountPercent", "count")
    )

    for product, row in grouped.iterrows():
        values = product_totals[int(product)]
        values[0] += int(row["Records"])
        values[1] += float(row["Sales"])
        values[2] += float(row["SalesSquared"])
        values[3] += int(row["StockoutRecords"])
        values[4] += float(row["DiscountSum"])
        values[5] += int(row["DiscountCount"])

product_profile = pd.DataFrame([
    {
        "product_id": product,
        "Records": records,
        "AvgSales": sales / records,
        "StockoutRatePct": 100 * stockouts / records,
        "AvgDiscountPct": discount_sum / discount_count if discount_count else 0,
        "SalesCV": (
            np.sqrt(max(sales_squared / records - (sales / records) ** 2, 0))
            / (sales / records)
            if sales / records > 0 else 0
        )
    }
    for product, (records, sales, sales_squared, stockouts,
                  discount_sum, discount_count) in product_totals.items()
])

sales_cutoff = product_profile["AvgSales"].median()
stockout_cutoff = product_profile["StockoutRatePct"].median()
discount_cutoff = product_profile["AvgDiscountPct"].median()
volatility_cutoff = product_profile["SalesCV"].median()

high_sales = product_profile["AvgSales"] >= sales_cutoff
high_stockout = product_profile["StockoutRatePct"] >= stockout_cutoff
high_discount = product_profile["AvgDiscountPct"] >= discount_cutoff
high_volatility = product_profile["SalesCV"] >= volatility_cutoff

product_profile["Segment"] = np.select(
    [
        high_sales & high_stockout,
        high_stockout & high_discount,
        high_stockout & high_volatility,
        high_sales,
        high_stockout
    ],
    [
        "Replenishment priority",
        "Review promotion + stockout",
        "Volatile stockout watchlist",
        "Maintain availability",
        "Investigate low sales + stockout"
    ],
    default="Lower priority"
)

segment_summary = (
    product_profile.groupby("Segment")
    .agg(
        Products=("product_id", "count"),
        AvgSales=("AvgSales", "mean"),
        AvgStockoutRatePct=("StockoutRatePct", "mean"),
        AvgDiscountPct=("AvgDiscountPct", "mean"),
        AvgSalesCV=("SalesCV", "mean")
    )
    .sort_values("Products", ascending=False)
)

print("Median cutoffs used:")
print("Average sales:", round(sales_cutoff, 3))
print("Stockout rate (%):", round(stockout_cutoff, 2))
print("Average discount (%):", round(discount_cutoff, 2))
print("Sales volatility (CV):", round(volatility_cutoff, 3))

print("\nProduct segments:")
display(segment_summary.round(2))

print("\nTop replenishment-priority products:")
display(
    product_profile[
        product_profile["Segment"] == "Replenishment priority"
    ].sort_values(
        ["StockoutRatePct", "AvgSales"],
        ascending=False
    )[
        [
            "product_id", "Records", "AvgSales", "StockoutRatePct",
            "AvgDiscountPct", "SalesCV"
        ]
    ].head(10).round(2)
)

Median cutoffs used:
Average sales: 0.521
Stockout rate (%): 37.78
Average discount (%): 5.13
Sales volatility (CV): 0.602

Product segments:


,Products,AvgSales,AvgStockoutRatePct,AvgDiscountPct,AvgSalesCV
Segment,,,,,
Maintain availability,230,0.89,29.49,8.13,0.71
Replenishment priority,203,1.11,50.46,8.58,0.63
Lower priority,202,0.47,28.81,7.72,0.74
Review promotion + stockout,107,0.47,51.06,10.83,0.64
Investigate low sales + stockout,96,0.46,51.81,1.97,0.48
Volatile stockout watchlist,27,0.47,49.21,3.07,0.76



Top replenishment-priority products:


,product_id,Records,AvgSales,StockoutRatePct,AvgDiscountPct,SalesCV
93,224,5310,0.75,99.23,3.37,0.42
432,227,990,0.87,99.09,0.44,0.39
442,752,720,0.61,98.47,2.77,0.44
94,228,2970,0.72,97.95,7.97,0.44
277,226,2880,0.80,97.60,1.12,0.39
333,313,2430,0.55,96.79,0.93,0.35
411,312,180,1.00,95.00,0.26,0.42
440,225,180,0.58,95.00,3.70,0.40
542,589,4500,10.79,81.18,99.91,0.41
134,379,10710,0.68,80.90,4.03,0.88


### Segmentation interpretation and findings

The cutoffs are medians across products: average sales = 0.521, stockout rate = 37.78%, average discount = 5.13%, and sales CV = 0.602. They create relative groups within this dataset, not universal business thresholds.

- **Replenishment priority:** 203 products had above-median sales and stockout rates. Their average product-level sales were 1.11 and average stockout rate was 50.46%. Products 224, 228, and 226 are examples with stockout rates of 99.23%, 97.95%, and 97.60%, respectively. Review these products for replenishment, then confirm the specific stores using the product-store analysis.
- **Review promotion + stockout:** 107 products had above-median stockout rates and discounts. Their mean stockout rate was 51.06% and mean discount was 10.83%. Review whether discounts coincide with availability problems; this association does not prove discounts caused stockouts.
- **Volatile stockout watchlist:** 27 products had above-median stockout rates and sales volatility. Their average sales CV was 0.76, compared with the 0.602 median. These products may need closer monitoring because sales vary more.
- **Maintain availability:** 230 products had above-median sales but below-median stockout rates. Maintain service levels and monitor changes in availability.
- **Investigate low sales + stockout:** 96 products had below-median sales but above-median stockout rates. Investigate local demand, assortment, and replenishment before increasing stock automatically.
- **Lower priority:** 202 products were below the sales and stockout medians. This is a relative prioritization label, not a recommendation to stop monitoring them.

Product 589 merits a separate check: its average discount is 99.91% under the dataset's discount definition. Verify the underlying discount values and their business meaning before drawing conclusions about its promotion performance.

The values in the segment summary are averages across products within each segment, not weighted averages across all product-store-day records. `SalesCV` is the coefficient of variation (standard deviation divided by mean); a higher value indicates more variable sales. `sale_amount` is a normalized sales measure, not necessarily monetary revenue. These segments guide review and do not prove that discounts or stockouts caused sales changes.